In [1]:
import os
import shutil
from pathlib import Path

import pandas as pd

from PIL import Image
from sklearn.model_selection import train_test_split

In [2]:
# ORIGINAL DATASET

DATASET_PATH = "./brain_tumor/Dataset/classification_data"

TRAIN_PATH = os.path.join(DATASET_PATH, "Training")
TEST_PATH = os.path.join(DATASET_PATH, "Testing")

# OUTPUT DATASET

OUTPUT_PATH = "./brain_tumor/Dataset/classification_data/Processed"

TRAIN_OUTPUT = os.path.join(OUTPUT_PATH, "train")
VAL_OUTPUT = os.path.join(OUTPUT_PATH, "val")
TEST_OUTPUT = os.path.join(OUTPUT_PATH, "test")

# VALIDATION SPLIT

VAL_SIZE = 0.20

# RANDOM SEED

RANDOM_STATE = 42

In [3]:
classes = sorted(
    [
        folder
        for folder in os.listdir(TRAIN_PATH)
        if os.path.isdir(
            os.path.join(TRAIN_PATH, folder)
        )
    ]
)

print("Classes Found:")
print(classes)

Classes Found:
['glioma', 'meningioma', 'notumor', 'pituitary']


In [4]:
for split_folder in [
    TRAIN_OUTPUT,
    VAL_OUTPUT,
    TEST_OUTPUT
]:

    os.makedirs(split_folder, exist_ok=True)

    for cls in classes:

        os.makedirs(
            os.path.join(
                split_folder,
                cls
            ),
            exist_ok=True
        )

print("Folders Created")

Folders Created


In [5]:
train_split = {}
val_split = {}

for cls in classes:

    class_folder = os.path.join(
        TRAIN_PATH,
        cls
    )

    images = [
        img
        for img in os.listdir(class_folder)
        if img.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp"
            )
        )
    ]

    train_imgs, val_imgs = train_test_split(
        images,
        test_size=VAL_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    train_split[cls] = train_imgs
    val_split[cls] = val_imgs

    print(
        f"{cls}: "
        f"Train={len(train_imgs)} "
        f"Val={len(val_imgs)}"
    )

glioma: Train=2033 Val=509
meningioma: Train=1573 Val=394
notumor: Train=1120 Val=280
pituitary: Train=1716 Val=430


In [6]:
csv_records = []

for cls in classes:

    for idx, image_name in enumerate(
        train_split[cls],
        start=1
    ):

        old_path = os.path.join(
            TRAIN_PATH,
            cls,
            image_name
        )

        new_name = (
            f"{cls}_train_{idx:05d}.png"
        )

        new_path = os.path.join(
            TRAIN_OUTPUT,
            cls,
            new_name
        )

        img = Image.open(old_path)

        img.save(
            new_path,
            format="PNG"
        )

        csv_records.append([
            image_name,
            old_path,
            new_name,
            new_path,
            cls
        ])

print("Training Processed")

Training Processed


In [7]:
for cls in classes:

    for idx, image_name in enumerate(
        val_split[cls],
        start=1
    ):

        old_path = os.path.join(
            TRAIN_PATH,
            cls,
            image_name
        )

        new_name = (
            f"{cls}_val_{idx:05d}.png"
        )

        new_path = os.path.join(
            VAL_OUTPUT,
            cls,
            new_name
        )

        img = Image.open(old_path)

        img.save(
            new_path,
            format="PNG"
        )

        csv_records.append([
            image_name,
            old_path,
            new_name,
            new_path,
            cls
        ])

print("Validation Processed")

Validation Processed


In [8]:
for cls in classes:

    class_folder = os.path.join(
        TEST_PATH,
        cls
    )

    images = [
        img
        for img in os.listdir(class_folder)
        if img.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp"
            )
        )
    ]

    for idx, image_name in enumerate(
        images,
        start=1
    ):

        old_path = os.path.join(
            class_folder,
            image_name
        )

        new_name = (
            f"{cls}_test_{idx:05d}.png"
        )

        new_path = os.path.join(
            TEST_OUTPUT,
            cls,
            new_name
        )

        img = Image.open(old_path)

        img.save(
            new_path,
            format="PNG"
        )

        csv_records.append([
            image_name,
            old_path,
            new_name,
            new_path,
            cls
        ])

print("Testing Processed")

Testing Processed


In [9]:
dataset_df = pd.DataFrame(
    csv_records,
    columns=[
        "old_filename",
        "old_path",
        "new_filename",
        "new_path",
        "class"
    ]
)

csv_path = os.path.join(
    OUTPUT_PATH,
    "dataset.csv"
)

dataset_df.to_csv(
    csv_path,
    index=False
)

print("CSV Saved")
dataset_df.head()

CSV Saved


,old_filename,old_path,new_filename,new_path,class
0,1673.png,./brain_tumor/Dataset/classification_data\Trai...,glioma_train_00001.png,./brain_tumor/Dataset/classification_data/Proc...,glioma
1,Tr-gl_807.jpg,./brain_tumor/Dataset/classification_data\Trai...,glioma_train_00002.png,./brain_tumor/Dataset/classification_data/Proc...,glioma
2,Tr-gl_707.jpg,./brain_tumor/Dataset/classification_data\Trai...,glioma_train_00003.png,./brain_tumor/Dataset/classification_data/Proc...,glioma
3,Tr-gl_20.jpg,./brain_tumor/Dataset/classification_data\Trai...,glioma_train_00004.png,./brain_tumor/Dataset/classification_data/Proc...,glioma
4,Tr-gl_918.jpg,./brain_tumor/Dataset/classification_data\Trai...,glioma_train_00005.png,./brain_tumor/Dataset/classification_data/Proc...,glioma


In [10]:
dataset_df.duplicated().sum()

0